# Point Cloud Analysis for ACT Policy Episodes

This notebook provides comprehensive tools for analyzing robotics episode data from HDF5 files, including:
- Camera feeds (RGB + Depth)
- Joint trajectories and actions  
- 3D point cloud visualization
- Action type comparison (raw, processed, QP-filtered)
- Strategic timeframe analysis

Generated with optimized point cloud processing for QP filter analysis.

In [ ]:
# Setup and Import Libraries
import os
import h5py
import numpy as np
import matplotlib.pyplot as plt
import mediapy as media

# Check for interactive plotting capabilities
try:
    import ipywidgets
    print("✓ ipywidgets available - interactive plots enabled")
    %matplotlib widget
    interactive_available = True
except ImportError:
    print("⚠ ipywidgets not available - using static plots")
    print("Install with: pip install ipywidgets")
    %matplotlib inline
    interactive_available = False

print(f"📚 Libraries loaded successfully!")
print(f"🎛 Interactive plotting: {'Enabled' if interactive_available else 'Static mode'}")

# Test matplotlib configuration
fig, ax = plt.subplots(1, 1, figsize=(6, 4))
test_data = np.sin(np.linspace(0, 2*np.pi, 100))
ax.plot(test_data)
ax.set_title('✓ Matplotlib Working')
plt.show()

In [ ]:
# Load HDF5 Data Function
def load_hdf5(dataset_path):
    """
    Load robotics episode data from HDF5 file with support for point clouds and multiple action types.
    
    Returns:
        qpos, qvel, action_data, image_dict, attrs, pointcloud_dict
    """
    if not os.path.isfile(dataset_path):
        print(f'Dataset does not exist at \n{dataset_path}\n')
        return None, None, None, None, None, None
    
    with h5py.File(dataset_path, 'r') as root:
        # Load joint data
        qpos = root['/observations/qpos'][()]
        qvel = root['/observations/qvel'][()]
        
        # Load camera images
        image_dict = dict()
        for cam_name in root[f'/observations/images/'].keys():
            image_dict[cam_name] = root[f'/observations/images/{cam_name}'][()]
        
        # Load all action types for QP filter analysis
        action_data = {}
        action_data['action'] = root['/action'][()]  # Final QP-filtered actions
        
        # Load additional action types if available
        if '/action_raw' in root:
            action_data['action_raw'] = root['/action_raw'][()]
        if '/action_processed' in root:
            action_data['action_processed'] = root['/action_processed'][()]
        if '/action_filtered' in root:
            action_data['action_filtered'] = root['/action_filtered'][()]
            
        # Load point cloud data if available
        pointcloud_dict = {}
        if 'observations/pointclouds' in root:
            for cam_name in root['observations/pointclouds'].keys():
                pointcloud_dict[cam_name] = {
                    'points_camera': root[f'observations/pointclouds/{cam_name}/points_camera'][()],
                    'points_world': root[f'observations/pointclouds/{cam_name}/points_world'][()], 
                    'num_points': root[f'observations/pointclouds/{cam_name}/num_points'][()]
                }
            
        # Load metadata attributes
        attrs = {}
        for key in root.attrs.keys():
            attrs[key] = root.attrs[key]
    
    return qpos, qvel, action_data, image_dict, attrs, pointcloud_dict

def load_pointcloud_from_hdf5(dataset_path, camera_name='cam_high', timestep=0):
    """
    Load point cloud data from HDF5 file for specific camera and timestep.
    """
    with h5py.File(dataset_path, 'r') as root:
        points_camera = root[f'observations/pointclouds/{camera_name}/points_camera'][timestep]
        points_world = root[f'observations/pointclouds/{camera_name}/points_world'][timestep]
        num_points = root[f'observations/pointclouds/{camera_name}/num_points'][timestep]
    
    return points_camera, points_world, num_points

print("✓ HDF5 loading functions ready")

In [ ]:
# Display Camera Data with Video Playback
def process_depth_for_visualization(depth_mm, manipulation_focus=True):
    """
    Convert depth images from millimeters to visualization-ready format.
    
    Args:
        depth_mm: Depth images in millimeters (uint16)
        manipulation_focus: If True, focus on A4-sized working area (0.3-1.0m range)
    
    Returns:
        depth_vis: Normalized depth for visualization (0-255, closer=brighter)
        depth_stats: Dictionary with depth statistics
    """
    # Convert from mm to meters
    depth_m = depth_mm.astype(np.float32) / 1000.0
    
    if manipulation_focus:
        # Focus on A4 working area: 0.3-1.0m range
        min_depth, max_depth = 0.3, 1.0
        title_suffix = "(manipulation zone)"
    else:
        # Use full range
        min_depth, max_depth = depth_m.min(), depth_m.max()
        title_suffix = "(full range)"
    
    # Clip and normalize depth
    depth_clipped = np.clip(depth_m, min_depth, max_depth)
    depth_normalized = 1.0 - (depth_clipped - min_depth) / (max_depth - min_depth)
    depth_vis = (depth_normalized * 255).astype(np.uint8)
    
    # Calculate statistics
    stats = {
        'min_depth_m': depth_m.min(),
        'max_depth_m': depth_m.max(), 
        'mean_depth_m': depth_m.mean(),
        'manipulation_mean_m': np.mean(depth_m[(depth_m >= 0.3) & (depth_m <= 1.0)]),
        'title_suffix': title_suffix,
        'vis_range': f"{min_depth:.1f}-{max_depth:.1f}m"
    }
    
    return depth_vis, stats

def display_camera_data(image_dict, fps=30):
    """Display both RGB and depth camera data with appropriate processing."""
    
    rgb_cameras = []
    depth_cameras = []
    
    # Separate RGB and depth cameras
    for cam_name, image_data in image_dict.items():
        if cam_name.endswith('_depth'):
            depth_cameras.append((cam_name, image_data))
        else:
            rgb_cameras.append((cam_name, image_data))
    
    print(f"📷 Found {len(rgb_cameras)} RGB cameras and {len(depth_cameras)} depth cameras")
    
    # Display RGB cameras
    for cam_name, image_list in rgb_cameras:
        print(f"\n🎬 RGB Camera: {cam_name}")
        print(f"   Shape: {image_list.shape}, dtype: {image_list.dtype}")
        media.show_video(image_list, fps=fps)
    
    # Display depth cameras  
    for cam_name, depth_list in depth_cameras:
        print(f"\n📏 Depth Camera: {cam_name}")
        print(f"   Shape: {depth_list.shape}, dtype: {depth_list.dtype}")
        
        # Process first frame for statistics
        depth_vis, stats = process_depth_for_visualization(depth_list, manipulation_focus=True)
        
        print(f"   Depth range: {stats['min_depth_m']:.3f} - {stats['max_depth_m']:.3f}m")
        print(f"   Mean depth: {stats['mean_depth_m']:.3f}m")
        print(f"   Manipulation area mean: {stats['manipulation_mean_m']:.3f}m")
        print(f"   Visualization range: {stats['vis_range']}")
        
        # Show depth video with processed visualization
        media.show_video(depth_vis, fps=fps)

print("✓ Camera display functions ready")

In [ ]:
# Plot Joint Trajectories
def plot_joint_trajectories(qpos, action_data, joint_names=None):
    """
    Plot joint positions and actions over time.
    
    Args:
        qpos: Joint positions array
        action_data: Dictionary of action types
        joint_names: List of joint names (defaults to Joint 1, 2, etc.)
    """
    n_joints = qpos.shape[1]
    
    if joint_names is None:
        joint_names = [f'Joint {i+1}' for i in range(n_joints)]
        if n_joints == 7:  # Assume 7th is gripper
            joint_names[-1] = 'Gripper'
    
    # Plot joint positions
    plt.figure(figsize=(15, 8))
    
    plt.subplot(2, 1, 1)
    for i in range(n_joints):
        plt.plot(qpos[:, i], label=joint_names[i], linewidth=2, alpha=0.8)
    plt.xlabel('Timestep')
    plt.ylabel('Position [rad] / [0-1]')
    plt.title('Joint Positions Over Time')
    plt.legend()
    plt.grid(True, alpha=0.3)
    
    # Plot actions (use main action if multiple types available)
    plt.subplot(2, 1, 2)
    action_to_plot = action_data.get('action', list(action_data.values())[0])
    
    for i in range(action_to_plot.shape[1]):
        plt.plot(action_to_plot[:, i], label=joint_names[i], linewidth=2, alpha=0.8)
    plt.xlabel('Timestep')
    plt.ylabel('Action [rad] / [0-1]')
    plt.title('Joint Actions Over Time')
    plt.legend()
    plt.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

def plot_action_comparison(action_data, attrs=None, joint_names=None):
    """
    Plot comparison of different action types to analyze QP filter effects.
    """
    if not action_data or len(action_data) < 2:
        print("⚠ Need at least 2 action types for comparison")
        return
    
    # Get main action for dimensions
    main_action = action_data.get('action', list(action_data.values())[0])
    n_joints = main_action.shape[1]
    
    if joint_names is None:
        joint_names = [f'Joint {i+1}' for i in range(n_joints)]
        if n_joints == 7:
            joint_names[-1] = 'Gripper'
    
    # Focus on key action types
    key_types = ['action_processed', 'action', 'action_filtered']
    available_types = [t for t in key_types if t in action_data]
    
    if not available_types:
        available_types = list(action_data.keys())[:3]  # Use first 3 available
    
    # Color scheme
    colors = ['#FF6B6B', '#4ECDC4', '#45B7D1', '#96CEB4', '#FFEAA7']
    
    # Create subplot grid
    n_cols = min(3, n_joints)
    n_rows = (n_joints + n_cols - 1) // n_cols
    
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, 4*n_rows))
    if n_joints == 1:
        axes = [axes]
    elif n_rows == 1:
        axes = axes.reshape(1, -1)
    
    timesteps = np.arange(len(main_action))
    
    for joint_idx in range(n_joints):
        row = joint_idx // n_cols
        col = joint_idx % n_cols
        ax = axes[row, col] if n_rows > 1 else axes[col]
        
        # Plot each available action type
        for i, action_type in enumerate(available_types):
            if action_type in action_data:
                ax.plot(timesteps, action_data[action_type][:, joint_idx],
                       color=colors[i % len(colors)], 
                       label=action_type.replace('action_', '').title(),
                       linewidth=1.5, alpha=0.8)
        
        ax.set_title(f'{joint_names[joint_idx]}')
        ax.set_xlabel('Timestep')
        ax.set_ylabel('Position [rad]')
        ax.grid(True, alpha=0.3)
        ax.legend()
    
    # Hide unused subplots
    for joint_idx in range(n_joints, n_rows * n_cols):
        row = joint_idx // n_cols
        col = joint_idx % n_cols
        ax = axes[row, col] if n_rows > 1 else axes[col]
        ax.set_visible(False)
    
    # Add title with metadata
    title = "Action Type Comparison"
    if attrs:
        obstacle_height = attrs.get('obstacle_height', 'N/A')
        qp_enabled = attrs.get('qp_filter_enabled', False)
        title += f" - Obstacle: {obstacle_height}m, QP Filter: {'Yes' if qp_enabled else 'No'}"
    
    fig.suptitle(title, fontsize=16, y=0.98)
    plt.tight_layout()
    plt.subplots_adjust(top=0.93)
    plt.show()

print("✓ Joint trajectory plotting functions ready")

In [ ]:
# Analyze Point Cloud Data
def display_pointcloud_data(pointcloud_dict, timestep=0):
    """
    Display point cloud data with multiple viewing angles and statistics.
    """
    if not pointcloud_dict:
        print("⚠ No point cloud data available")
        return
    
    camera_name = list(pointcloud_dict.keys())[0]
    pc_data = pointcloud_dict[camera_name]
    
    points_camera = pc_data['points_camera'][timestep]
    points_world = pc_data['points_world'][timestep] 
    num_points = pc_data['num_points'][timestep]
    
    print(f"📊 Point Cloud Analysis (Timestep {timestep})")
    print(f"   Camera: {camera_name}")
    print(f"   Points: {num_points}")
    print(f"   Camera coords range: X[{points_camera[:, 0].min():.3f}, {points_camera[:, 0].max():.3f}] "
          f"Y[{points_camera[:, 1].min():.3f}, {points_camera[:, 1].max():.3f}] "
          f"Z[{points_camera[:, 2].min():.3f}, {points_camera[:, 2].max():.3f}]")
    
    # Subsample for visualization
    max_points = 3000
    if len(points_camera) > max_points:
        indices = np.random.choice(len(points_camera), max_points, replace=False)
        points = points_camera[indices]
    else:
        points = points_camera
    
    # Create multiple views
    fig = plt.figure(figsize=(15, 10))
    
    # 3D views
    views = [
        (20, 45, "Default View"),
        (90, 0, "Top-Down View"), 
        (0, 0, "Side View"),
        (0, 90, "Front View")
    ]
    
    for i, (elev, azim, title) in enumerate(views):
        ax = fig.add_subplot(2, 3, i+1, projection='3d')
        scatter = ax.scatter(points[:, 0], points[:, 1], points[:, 2], 
                           c=points[:, 2], s=1, cmap='viridis')
        ax.set_title(title)
        ax.set_xlabel('X (width)')
        ax.set_ylabel('Y (depth)')
        ax.set_zlabel('Z (height)')
        ax.view_init(elev=elev, azim=azim)
    
    # 2D projections
    ax5 = fig.add_subplot(2, 3, 5)
    ax5.scatter(points[:, 0], points[:, 1], c=points[:, 2], s=1, cmap='viridis')
    ax5.set_title('XY Projection (Top View)')
    ax5.set_xlabel('X (width)')
    ax5.set_ylabel('Y (depth)')
    ax5.set_aspect('equal')
    
    ax6 = fig.add_subplot(2, 3, 6)
    ax6.scatter(points[:, 0], points[:, 2], c=points[:, 1], s=1, cmap='viridis')
    ax6.set_title('XZ Projection (Side View)')
    ax6.set_xlabel('X (width)')
    ax6.set_ylabel('Z (height)')
    ax6.set_aspect('equal')
    
    plt.tight_layout()
    plt.show()
    
    return points_camera, points_world, num_points

def analyze_strategic_timeframes(dataset_path, max_timestep=250, num_frames=5):
    """
    Analyze point clouds at strategic timeframes during robot manipulation.
    """
    with h5py.File(dataset_path, 'r') as f:
        total_timesteps = f['action'].shape[0]
        if 'observations/pointclouds' not in f:
            print("⚠ No point cloud data in this episode")
            return None
        
        available_cameras = list(f['observations/pointclouds/'].keys())
    
    print(f"📊 Strategic Timeframe Analysis:")
    print(f"   Total timesteps: {total_timesteps}")
    print(f"   Analyzing first {min(max_timestep, total_timesteps)} timesteps")
    print(f"   Available cameras: {available_cameras}")
    
    # Calculate strategic timeframes
    end_step = min(max_timestep, total_timesteps)
    strategic_steps = np.linspace(0, end_step-1, num_frames, dtype=int)
    
    camera_name = available_cameras[0]
    
    # Create visualization grid
    n_cols = min(5, num_frames)
    n_rows = (num_frames + n_cols - 1) // n_cols
    
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, 3*n_rows))
    
    # Fix axis indexing - handle single row/column cases properly
    if n_rows == 1 and n_cols == 1:
        axes = np.array([[axes]])  # Make it 2D
    elif n_rows == 1:
        axes = axes.reshape(1, -1)  # Single row, multiple columns
    elif n_cols == 1:
        axes = axes.reshape(-1, 1)  # Multiple rows, single column
    
    point_clouds = []
    
    for i, timestep in enumerate(strategic_steps):
        row = i // n_cols
        col = i % n_cols
        
        try:
            points_camera, points_world, num_points = load_pointcloud_from_hdf5(
                dataset_path, camera_name=camera_name, timestep=timestep
            )
            
            point_clouds.append((timestep, points_camera, num_points))
            
            # Subsample for performance
            max_points = 2000
            if len(points_camera) > max_points:
                indices = np.random.choice(len(points_camera), max_points, replace=False)
                points = points_camera[indices]
            else:
                points = points_camera
            
            # Get the correct axis - always use 2D indexing now
            ax = axes[row, col]
            
            # Top-down view (invert Y for correct orientation)
            scatter = ax.scatter(points[:, 0], -points[:, 1], c=points[:, 2], s=2, cmap='viridis')
            ax.set_title(f'Step {timestep}\n({num_points} pts)')
            ax.set_xlabel('X (width)')
            ax.set_ylabel('Y (depth)')
            ax.set_aspect('equal')
                
        except Exception as e:
            print(f"⚠ Error at timestep {timestep}: {e}")
            print(f"   Debug info: row={row}, col={col}, axes.shape={axes.shape}")
    
    # Hide empty subplots
    for i in range(len(strategic_steps), n_rows * n_cols):
        row = i // n_cols
        col = i % n_cols
        if row < n_rows and col < n_cols:
            ax = axes[row, col]
            ax.axis('off')
    
    plt.suptitle(f'Point Cloud Evolution: Strategic Timeframes', fontsize=16)
    plt.tight_layout()
    plt.show()
    
    return point_clouds

print("✓ Point cloud analysis functions ready")

In [ ]:
# Create Manipulation Sequence Visualization
def create_manipulation_sequence(dataset_path, max_timestep=250, step_size=20):
    """
    Create a sequence of point cloud snapshots showing manipulation progression.
    """
    with h5py.File(dataset_path, 'r') as f:
        total_timesteps = f['action'].shape[0]
        if 'observations/pointclouds' not in f:
            print("⚠ No point cloud data available")
            return None
        camera_name = list(f['observations/pointclouds/'].keys())[0]
    
    end_step = min(max_timestep, total_timesteps)
    timesteps = list(range(0, end_step, step_size))
    
    print(f"📽 Creating manipulation sequence:")
    print(f"   Timesteps: {timesteps}")
    print(f"   Camera: {camera_name}")
    
    # Calculate grid dimensions
    n_cols = 5
    n_rows = (len(timesteps) + n_cols - 1) // n_cols
    
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, 3*n_rows))
    
    # Fix axis indexing - handle single row/column cases properly
    if n_rows == 1 and n_cols == 1:
        axes = np.array([[axes]])  # Make it 2D
    elif n_rows == 1:
        axes = axes.reshape(1, -1)  # Single row, multiple columns
    elif n_cols == 1:
        axes = axes.reshape(-1, 1)  # Multiple rows, single column
    
    sequence_data = []
    
    for i, timestep in enumerate(timesteps):
        row = i // n_cols
        col = i % n_cols
        
        try:
            points_camera, points_world, num_points = load_pointcloud_from_hdf5(
                dataset_path, camera_name=camera_name, timestep=timestep
            )
            
            sequence_data.append((timestep, points_camera, num_points))
            
            # Subsample for performance
            max_points = 2000
            if len(points_camera) > max_points:
                indices = np.random.choice(len(points_camera), max_points, replace=False)
                points = points_camera[indices]
            else:
                points = points_camera
            
            # Get the correct axis - always use 2D indexing now
            ax = axes[row, col]
            
            # Top-down view with correct orientation
            scatter = ax.scatter(points[:, 0], -points[:, 1], c=points[:, 2], s=2, cmap='viridis')
            ax.set_title(f'Step {timestep}')
            ax.set_xlabel('X (width)')
            ax.set_ylabel('Y (depth)')
            ax.set_aspect('equal')
            
            # Add point count annotation
            ax.text(0.02, 0.98, f'{num_points} pts', transform=ax.transAxes,
                   verticalalignment='top', bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))
        
        except Exception as e:
            print(f"⚠ Error at timestep {timestep}: {e}")
            print(f"   Debug info: row={row}, col={col}, axes.shape={axes.shape}")
    
    # Hide empty subplots
    for i in range(len(timesteps), n_rows * n_cols):
        row = i // n_cols
        col = i % n_cols
        if row < n_rows and col < n_cols:
            ax = axes[row, col]
            ax.axis('off')
    
    plt.suptitle(f'Manipulation Sequence: Steps 0-{end_step} (every {step_size} steps)', fontsize=16)
    plt.tight_layout()
    plt.show()
    
    return sequence_data

print("✓ Manipulation sequence visualization ready")

## Example Usage
Load and analyze an episode with all available data types:

In [ ]:
# Example: Load and analyze an episode
# Update this path to your episode file
data_file = '/home/hk/Documents/ACT_Shaka/act-main/act/dataset_dir/real_episodes_source/episode_37.hdf5'

# Load all data types
qpos, qvel, action_data, image_dict, attrs, pointcloud_dict = load_hdf5(data_file)

if qpos is not None:
    print(f"📊 Episode Statistics:")
    print(f"   Duration: {len(action_data['action'])} timesteps ({len(action_data['action'])*0.02:.1f}s at 50Hz)")
    print(f"   Action space: {action_data['action'].shape}")
    print(f"   Joint positions: {qpos.shape}")
    print(f"   Available cameras: {list(image_dict.keys())}")
    print(f"   Point cloud cameras: {list(pointcloud_dict.keys()) if pointcloud_dict else 'None'}")
    
    if attrs:
        print(f"\n📋 Episode Metadata:")
        for key, value in attrs.items():
            if key == 'obstacle_height' and value is not None:
                print(f"   {key}: {value:.3f}m")
            else:
                print(f"   {key}: {value}")
    
    print(f"\n🎯 Action Types Available:")
    for action_type, data in action_data.items():
        print(f"   {action_type}: {data.shape}")
else:
    print("❌ Failed to load episode data. Check file path.")

In [ ]:
# Display camera feeds (RGB and depth videos)
if qpos is not None and image_dict:
    display_camera_data(image_dict, fps=30)

In [ ]:
# Plot joint trajectories and action comparison
if qpos is not None:
    # Define joint names for 7-DOF system (6 joints + gripper)
    joint_names = ['Shoulder Pan', 'Shoulder Lift', 'Elbow', 'Wrist 1', 'Wrist 2', 'Wrist 3', 'Gripper']
    
    # Plot basic trajectories
    plot_joint_trajectories(qpos, action_data, joint_names[:action_data['action'].shape[1]])
    
    # Compare different action types if available
    if len(action_data) > 1:
        plot_action_comparison(action_data, attrs, joint_names[:action_data['action'].shape[1]])

In [ ]:
# Analyze point cloud data (if available)
if pointcloud_dict:
    # Display single timestep point cloud with multiple views
    points_camera, points_world, num_points = display_pointcloud_data(pointcloud_dict, timestep=0)
    
    # Analyze strategic timeframes during manipulation
    strategic_data = analyze_strategic_timeframes(data_file, max_timestep=250, num_frames=5)
    
    # Create manipulation sequence animation
    sequence_data = create_manipulation_sequence(data_file, max_timestep=250, step_size=25)
else:
    print("⚠ No point cloud data available in this episode")
    print("  To generate point clouds, run: python generate_pointclouds.py --dataset_dir <path> --task_name <task> --target_points 1500")

## Quick Analysis Templates

Copy and modify these code blocks for different episode files:

In [ ]:
# Template: Quick episode analysis
def quick_analysis(episode_path):
    """Quick analysis of any episode file."""
    print(f"🔍 Quick Analysis: {episode_path}")
    
    # Load data
    qpos, qvel, action_data, image_dict, attrs, pointcloud_dict = load_hdf5(episode_path)
    
    if qpos is None:
        print("❌ Failed to load episode")
        return
    
    # Basic stats
    print(f"   Duration: {len(action_data['action'])} timesteps")
    print(f"   Cameras: {list(image_dict.keys())}")
    print(f"   Point clouds: {'Yes' if pointcloud_dict else 'No'}")
    print(f"   Action types: {list(action_data.keys())}")
    
    # Show first camera frame if available
    if image_dict:
        cam_name = list(image_dict.keys())[0]
        plt.figure(figsize=(8, 6))
        plt.imshow(image_dict[cam_name][0])
        plt.title(f"First frame: {cam_name}")
        plt.axis('off')
        plt.show()
    
    return qpos, qvel, action_data, image_dict, attrs, pointcloud_dict

# Example usage:
# quick_analysis('/path/to/your/episode.hdf5')

In [ ]:
# Template: Batch analysis of multiple episodes
def batch_analysis(episode_dir, pattern="episode_*.hdf5"):
    """Analyze multiple episodes in a directory."""
    import glob
    
    episode_files = glob.glob(os.path.join(episode_dir, pattern))
    episode_files.sort()
    
    print(f"📊 Batch Analysis: {len(episode_files)} episodes found")
    
    results = []
    for episode_file in episode_files[:5]:  # Analyze first 5 episodes
        print(f"\n📁 Analyzing: {os.path.basename(episode_file)}")
        
        qpos, qvel, action_data, image_dict, attrs, pointcloud_dict = load_hdf5(episode_file)
        
        if qpos is not None:
            episode_info = {
                'file': episode_file,
                'duration': len(action_data['action']),
                'has_pointcloud': bool(pointcloud_dict),
                'action_types': list(action_data.keys()),
                'cameras': list(image_dict.keys()),
                'obstacle_height': attrs.get('obstacle_height', None) if attrs else None
            }
            results.append(episode_info)
            print(f"   ✓ Duration: {episode_info['duration']} steps")
            print(f"   ✓ Point clouds: {'Yes' if episode_info['has_pointcloud'] else 'No'}")
    
    return results

# Example usage:
# results = batch_analysis('/home/hk/Documents/ACT_Shaka/act-main/act/ckpt_sim_pick_cube_teleop/eval_data/')

In [ ]:
# Compare Ground Truth vs ReactivePointNet Rollout - Residual Analysis
def compare_residuals_ground_truth_vs_rollout():
    """
    Compare residual norms between ground truth episode and ReactivePointNet rollout.
    Analyzes data coverage near contact by examining residual patterns.
    """
    print("🔍 RESIDUAL ANALYSIS: Ground Truth vs ReactivePointNet Rollout")
    print("=" * 80)
    
    # File paths
    ground_truth_file = '/home/hk/Documents/ACT_Shaka/act-main/act/ckpt_sim_pick_cube_teleop/eval_data/episode_33.hdf5'
    reactive_rollout_file = '/home/hk/Documents/ACT_Shaka/act-main/act/ckpt_sim_pick_cube_teleop/eval_data_oa/episode_45.hdf5'
    
    print(f"📂 Ground Truth: {os.path.basename(ground_truth_file)}")
    print(f"🤖 ReactivePointNet: {os.path.basename(reactive_rollout_file)}")
    
    # Load ground truth data
    qpos_gt, qvel_gt, action_data_gt, image_dict_gt, attrs_gt, pc_dict_gt = load_hdf5(ground_truth_file)
    
    # Load reactive rollout data  
    qpos_rn, qvel_rn, action_data_rn, image_dict_rn, attrs_rn, pc_dict_rn = load_hdf5(reactive_rollout_file)
    
    if qpos_gt is None or qpos_rn is None:
        print("❌ Failed to load episode data")
        return None
    
    # Print episode metadata
    print(f"\n📋 Episode Information:")
    print(f"Ground Truth - Success: {attrs_gt.get('success', 'N/A')}, Task Success: {attrs_gt.get('task_successful', 'N/A')}")
    print(f"ReactivePointNet - Success: {attrs_rn.get('task_successful', 'N/A')}, Obstacle Height: {attrs_rn.get('obstacle_height', 'N/A'):.3f}m")
    print(f"Ground Truth Duration: {len(qpos_gt)} timesteps")
    print(f"ReactivePointNet Duration: {len(qpos_rn)} timesteps")
    
    # Calculate residuals for ground truth (if QP data available)
    if 'action_processed' in action_data_gt and 'action' in action_data_gt:
        # Ground truth residual = action_final - action_processed (QP correction)
        gt_residuals = action_data_gt['action'] - action_data_gt['action_processed']
        gt_residual_norms = np.linalg.norm(gt_residuals[:, :6], axis=1)  # Only arm joints
        has_gt_residuals = True
        print("✓ Ground truth has QP residuals available")
    else:
        # No QP residuals in ground truth - compare raw vs processed actions
        if 'action_raw' in action_data_gt and 'action_processed' in action_data_gt:
            gt_residuals = action_data_gt['action_processed'] - action_data_gt['action_raw']
            gt_residual_norms = np.linalg.norm(gt_residuals[:, :6], axis=1)
            has_gt_residuals = True
            print("⚠ Ground truth using processed-raw residuals (no QP data)")
        else:
            gt_residual_norms = np.zeros(len(qpos_gt))
            has_gt_residuals = False
            print("⚠ No residual data available for ground truth")
    
    # Calculate ReactivePointNet residuals
    if 'action_processed' in action_data_rn and 'action' in action_data_rn:
        # ReactivePointNet residual = action_final - action_processed 
        rn_residuals = action_data_rn['action'] - action_data_rn['action_processed']
        rn_residual_norms = np.linalg.norm(rn_residuals[:, :6], axis=1)  # Only arm joints
        print("✓ ReactivePointNet residuals calculated")
    else:
        print("❌ No ReactivePointNet residual data available")
        return None
    
    # Ensure same length for comparison
    min_length = min(len(gt_residual_norms), len(rn_residual_norms))
    gt_norms = gt_residual_norms[:min_length]
    rn_norms = rn_residual_norms[:min_length]
    
    # Create visualization
    fig, axes = plt.subplots(3, 2, figsize=(16, 12))
    
    # Plot 1: Residual norms over time
    ax1 = axes[0, 0]
    timesteps = np.arange(min_length)
    ax1.plot(timesteps, gt_norms, 'b-', linewidth=2, alpha=0.8, label='Ground Truth ||Δq̇||')
    ax1.plot(timesteps, rn_norms, 'r-', linewidth=2, alpha=0.8, label='ReactivePointNet ||Δq̇||')
    ax1.set_xlabel('Timestep')
    ax1.set_ylabel('Residual Norm [rad/s]')
    ax1.set_title('Residual Norms Comparison Over Time')
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    # Plot 2: Cumulative residual comparison
    ax2 = axes[0, 1]
    gt_cumsum = np.cumsum(gt_norms)
    rn_cumsum = np.cumsum(rn_norms)
    ax2.plot(timesteps, gt_cumsum, 'b-', linewidth=2, alpha=0.8, label='Ground Truth (Cumulative)')
    ax2.plot(timesteps, rn_cumsum, 'r-', linewidth=2, alpha=0.8, label='ReactivePointNet (Cumulative)')
    ax2.set_xlabel('Timestep')
    ax2.set_ylabel('Cumulative Residual Norm')
    ax2.set_title('Cumulative Residual Comparison')
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    
    # Plot 3: Residual difference (ReactivePointNet - Ground Truth)
    ax3 = axes[1, 0]
    residual_diff = rn_norms - gt_norms
    ax3.plot(timesteps, residual_diff, 'purple', linewidth=2, alpha=0.8)
    ax3.axhline(0, color='black', linestyle='--', alpha=0.5)
    ax3.set_xlabel('Timestep')
    ax3.set_ylabel('Residual Difference [rad/s]')
    ax3.set_title('ReactivePointNet - Ground Truth Residuals')
    ax3.grid(True, alpha=0.3)
    
    # Plot 4: Histogram of residual norms
    ax4 = axes[1, 1]
    ax4.hist(gt_norms, bins=50, alpha=0.6, color='blue', label='Ground Truth', density=True)
    ax4.hist(rn_norms, bins=50, alpha=0.6, color='red', label='ReactivePointNet', density=True)
    ax4.set_xlabel('Residual Norm [rad/s]')
    ax4.set_ylabel('Density')
    ax4.set_title('Distribution of Residual Norms')
    ax4.legend()
    ax4.grid(True, alpha=0.3)
    
    # Plot 5: Contact detection - high residual regions
    ax5 = axes[2, 0]
    # Define contact threshold (high residual activity)
    contact_threshold = 0.05  # rad/s
    gt_contact = gt_norms > contact_threshold
    rn_contact = rn_norms > contact_threshold
    
    ax5.fill_between(timesteps, 0, 1, where=gt_contact, alpha=0.3, color='blue', label='GT Contact Regions')
    ax5.fill_between(timesteps, 0, 1, where=rn_contact, alpha=0.3, color='red', label='RN Contact Regions')
    ax5.plot(timesteps, gt_norms / np.max(gt_norms), 'b-', alpha=0.8, linewidth=1)
    ax5.plot(timesteps, rn_norms / np.max(rn_norms), 'r-', alpha=0.8, linewidth=1)
    ax5.set_xlabel('Timestep')
    ax5.set_ylabel('Normalized Residual / Contact')
    ax5.set_title(f'Contact Detection (Threshold: {contact_threshold:.3f} rad/s)')
    ax5.legend()
    ax5.grid(True, alpha=0.3)
    
    # Plot 6: Joint space analysis - mean residual per joint
    ax6 = axes[2, 1]
    if has_gt_residuals and 'action_processed' in action_data_rn:
        joint_names = ['J1', 'J2', 'J3', 'J4', 'J5', 'J6']
        gt_joint_residuals = np.mean(np.abs(gt_residuals[:min_length, :6]), axis=0)
        rn_joint_residuals = np.mean(np.abs(rn_residuals[:min_length, :6]), axis=0)
        
        x = np.arange(6)
        width = 0.35
        ax6.bar(x - width/2, gt_joint_residuals, width, label='Ground Truth', alpha=0.8, color='blue')
        ax6.bar(x + width/2, rn_joint_residuals, width, label='ReactivePointNet', alpha=0.8, color='red')
        ax6.set_xlabel('Joint')
        ax6.set_ylabel('Mean |Residual| [rad/s]')
        ax6.set_title('Per-Joint Mean Residual Magnitude')
        ax6.set_xticks(x)
        ax6.set_xticklabels(joint_names)
        ax6.legend()
        ax6.grid(True, alpha=0.3)
    else:
        ax6.text(0.5, 0.5, 'Joint residuals not available', transform=ax6.transAxes, ha='center', va='center')
        ax6.set_title('Per-Joint Analysis (N/A)')
    
    plt.tight_layout()
    plt.show()
    
    # Statistical Analysis
    print(f"\n📊 STATISTICAL ANALYSIS:")
    print(f"Ground Truth Residuals:")
    print(f"  Mean: {np.mean(gt_norms):.6f} rad/s")
    print(f"  Max:  {np.max(gt_norms):.6f} rad/s")
    print(f"  Std:  {np.std(gt_norms):.6f} rad/s")
    print(f"  Contact periods (>{contact_threshold:.3f}): {np.sum(gt_contact)} timesteps ({np.sum(gt_contact)/len(gt_norms):.1%})")
    
    print(f"\nReactivePointNet Residuals:")
    print(f"  Mean: {np.mean(rn_norms):.6f} rad/s")
    print(f"  Max:  {np.max(rn_norms):.6f} rad/s") 
    print(f"  Std:  {np.std(rn_norms):.6f} rad/s")
    print(f"  Contact periods (>{contact_threshold:.3f}): {np.sum(rn_contact)} timesteps ({np.sum(rn_contact)/len(rn_norms):.1%})")
    
    # Data coverage analysis
    residual_ratio = np.mean(rn_norms) / np.mean(gt_norms) if np.mean(gt_norms) > 0 else float('inf')
    print(f"\n🎯 DATA COVERAGE ANALYSIS:")
    print(f"  Residual magnitude ratio (RN/GT): {residual_ratio:.3f}")
    print(f"  Correlation coefficient: {np.corrcoef(gt_norms, rn_norms)[0,1]:.3f}")
    
    # Log timestep-by-timestep data
    print(f"\n📝 TIMESTEP-BY-TIMESTEP RESIDUAL LOG (first 20 steps):")
    print(f"{'Step':<5} {'GT ||Δq̇||':<12} {'RN ||Δq̇||':<12} {'Difference':<10}")
    print("-" * 45)
    for t in range(min(20, min_length)):
        diff = rn_norms[t] - gt_norms[t]
        print(f"{t:<5} {gt_norms[t]:<12.6f} {rn_norms[t]:<12.6f} {diff:<10.6f}")
    
    if min_length > 20:
        print(f"... (showing first 20 of {min_length} timesteps)")
    
    return {
        'gt_norms': gt_norms,
        'rn_norms': rn_norms,
        'timesteps': timesteps,
        'residual_diff': residual_diff,
        'contact_threshold': contact_threshold,
        'gt_contact': gt_contact,
        'rn_contact': rn_contact
    }

# Run the comparison analysis
residual_analysis = compare_residuals_ground_truth_vs_rollout()